In [14]:
import pandas as pd
from pathlib import Path
import shutil


In [15]:
summary = pd.read_csv("../station_availability_summary_robust.csv")
summary

,station_folder,site_id,year,pm25_avail_pct,windspd_avail_pct,winddir_avail_pct
0,station_103_crri_mathura_road_delhi_imd_1hr,103,2021,96.80,0.00,0.00
1,station_104_burari_crossing_delhi_imd_1hr,104,2021,18.15,0.00,0.00
2,station_105_north_campus_du_delhi_imd_1hr,105,2021,93.15,0.00,0.00
3,station_106_igi_airport__t3__delhi_imd_1hr,106,2021,64.08,0.00,0.00
4,station_107_pusa_delhi_imd_1hr,107,2021,72.31,0.00,0.00
...,...,...,...,...,...,...
147,station_1562_sri_aurobindo_marg_delhi_dpcc_1hr,1562,2024,98.45,95.75,96.39
148,station_1563_pusa_delhi_dpcc_1hr,1563,2024,97.03,98.36,98.36
149,station_301_anand_vihar_delhi_dpcc_1hr,301,2024,82.81,84.88,86.70
150,station_5393_chandni_chowk_delhi_iitm_1hr,5393,2024,73.29,19.85,51.81


In [16]:
REQUIRED_YEARS = [2021,2022,2023,2024]
THRESH = 70.0


In [17]:
good_stations = []

for station, g in summary.groupby("station_folder"):
    ok = True
    
    for y in REQUIRED_YEARS:
        row = g[g["year"] == y]
        if row.empty:
            ok = False
            break
        
        r = row.iloc[0]
        if (r.pm25_avail_pct < THRESH or
            r.windspd_avail_pct < THRESH or
            r.winddir_avail_pct < THRESH):
            ok = False
            break
    
    if ok:
        good_stations.append(station)

print("Selected stations:", len(good_stations))
good_stations


Selected stations: 26


['station_113_shadipur_delhi_cpcb_1hr',
 'station_114_ihbas_dilshad_garden_delhi_cpcb_1hr',
 'station_115_nsit_dwarka_delhi_cpcb_1hr',
 'station_122_mandir_marg_delhi_dpcc_1hr',
 'station_124_r_k_puram_delhi_dpcc_1hr',
 'station_125_punjabi_bagh_delhi_dpcc_1hr',
 'station_1420_ashok_vihar_delhi_dpcc_1hr',
 'station_1421_dr__karni_singh_shooting_range_delhi_dpc',
 'station_1422_dwarka_sector_8_delhi_dpcc__1hr',
 'station_1423_jahangirpuri_delhi_dpcc_1hr',
 'station_1424_jawaharlal_nehru_stadium_delhi_dpcc_1hr',
 'station_1425_major_dhyan_chand_national_stadium_delhi',
 'station_1426_narela_delhi_dpcc_1hr',
 'station_1427_najafgarh_delhi_dpcc_1hr',
 'station_1428_okhla_phase_2_delhi_dpcc_1hr',
 'station_1429_nehru_nagar_delhi_dpcc_1hr',
 'station_1430_rohini_delhi_dpcc_1hr',
 'station_1431_patparganj_delhi_dpcc_1hr',
 'station_1432_sonia_vihar_delhi_dpcc_1hr',
 'station_1434_wazirpur_delhi_dpcc_1hr',
 'station_1435_vivek_vihar_delhi_dpcc_1hr',
 'station_1560_bawana_delhi_dpcc_1hr',
 'sta

In [18]:
pd.DataFrame({"station_folder": good_stations}).to_csv(
    "final_selected_stations.csv", index=False
)


In [19]:
RAW_DIR = Path("../station_data")
FINAL_DIR = Path("../final_station_data")
FINAL_DIR.mkdir(exist_ok=True)



In [20]:
for st in good_stations:
    src = RAW_DIR / st
    dst = FINAL_DIR / st

    if not src.exists():
        print("MISSING FOLDER:", src)
        continue

    if not dst.exists():
        shutil.copytree(src, dst)


In [21]:
def tidy_colname(s):
    import re
    s = str(s).lower()
    s = re.sub(r"[^\w]+","_",s)
    return s.strip("_")


In [22]:
PM_KEYS = ["pm2_5","pm25","pm"]
WS_KEYS = ["vws","ws","wind_speed","windspeed","speed","spd"]
WD_KEYS = ["wd","wind_dir","winddirection","direction","deg"]
TIME_KEYS = ["time","date","timestamp","datetime"]


In [23]:
def find_best(df, keys):
    best = None
    best_score = -1
    for c in df.columns:
        tc = tidy_colname(c)
        if any(k in tc for k in keys):
            score = pd.to_numeric(df[c], errors="coerce").notna().sum()
            if score > best_score:
                best = c
                best_score = score
    return best


In [24]:
FINAL_CLEAN = Path("final_station_data_clean")
FINAL_CLEAN.mkdir(exist_ok=True)


In [25]:
for station in good_stations:
    src_station = FINAL_DIR / station
    dst_station = FINAL_CLEAN / station
    dst_station.mkdir(exist_ok=True)
    
    for f in src_station.glob("*.csv"):
        df = pd.read_csv(f, low_memory=False)
        
        pm_col = find_best(df, PM_KEYS)
        ws_col = find_best(df, WS_KEYS)
        wd_col = find_best(df, WD_KEYS)
        t_col  = find_best(df, TIME_KEYS)
        
        if None in [pm_col, ws_col, wd_col, t_col]:
            print("SKIP (missing column):", f)
            continue
        
        clean = df[[t_col, pm_col, ws_col, wd_col]].rename(columns={
            t_col: "timestamp",
            pm_col: "PM 2.5",
            ws_col: "wind_speed",
            wd_col: "wind_dir"
        })
        
        clean.to_csv(dst_station / f.name, index=False)


In [27]:
DATA_DIR = Path("final_station_data_clean")
MAX_GAP_HOURS = 3   # only fill gaps up to 3 hours

def fill_short_gaps(df):
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp")
    df = df.set_index("timestamp")

    # create full hourly timeline
    full_index = pd.date_range(df.index.min(), df.index.max(), freq="h")
    df = df.reindex(full_index)

    # interpolate ONLY short gaps
    df["PM 2.5"] = df["PM 2.5"].interpolate(limit=MAX_GAP_HOURS)
    df["wind_speed"] = df["wind_speed"].interpolate(limit=MAX_GAP_HOURS)
    df["wind_dir"] = df["wind_dir"].interpolate(limit=MAX_GAP_HOURS)

    return df.reset_index().rename(columns={"index":"timestamp"})

for station in DATA_DIR.iterdir():
    if not station.is_dir():
        continue

    print("Station:", station.name)

    for f in station.glob("*.csv"):
        df = pd.read_csv(f)

        before = df.isna().sum()

        df_filled = fill_short_gaps(df)

        after = df_filled.isna().sum()

        df_filled.to_csv(f, index=False)

        print(f"  {f.name} | PM NaN: {before['PM 2.5']} → {after['PM 2.5']}")

Station: station_113_shadipur_delhi_cpcb_1hr
  2021.csv | PM NaN: 186 → 25
  2022.csv | PM NaN: 431 → 77
  2023.csv | PM NaN: 546 → 114
  2024.csv | PM NaN: 283 → 69
Station: station_114_ihbas_dilshad_garden_delhi_cpcb_1hr
  2021.csv | PM NaN: 569 → 185
  2022.csv | PM NaN: 986 → 430
  2023.csv | PM NaN: 790 → 500
  2024.csv | PM NaN: 408 → 158
Station: station_115_nsit_dwarka_delhi_cpcb_1hr
  2021.csv | PM NaN: 298 → 114
  2022.csv | PM NaN: 436 → 112
  2023.csv | PM NaN: 440 → 115
  2024.csv | PM NaN: 347 → 77
Station: station_122_mandir_marg_delhi_dpcc_1hr
  2021.csv | PM NaN: 303 → 167
  2022.csv | PM NaN: 279 → 171
  2023.csv | PM NaN: 437 → 292
  2024.csv | PM NaN: 1208 → 734
Station: station_124_r_k_puram_delhi_dpcc_1hr
  2021.csv | PM NaN: 465 → 292
  2022.csv | PM NaN: 184 → 87
  2023.csv | PM NaN: 399 → 283
  2024.csv | PM NaN: 435 → 216
Station: station_125_punjabi_bagh_delhi_dpcc_1hr
  2021.csv | PM NaN: 403 → 250
  2022.csv | PM NaN: 250 → 149
  2023.csv | PM NaN: 406 → 29

In [28]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("final_station_data_clean")

rows = []

for st in DATA_DIR.iterdir():
    if not st.is_dir():
        continue
    for f in st.glob("*.csv"):
        df = pd.read_csv(f)
        pm_pct = df["PM 2.5"].notna().mean()*100
        ws_pct = df["wind_speed"].notna().mean()*100
        wd_pct = df["wind_dir"].notna().mean()*100

        rows.append([st.name, f.name, round(pm_pct,2), round(ws_pct,2), round(wd_pct,2)])

final_avail = pd.DataFrame(rows, columns=[
    "station","file","pm25_pct","windspd_pct","winddir_pct"
])

final_avail.to_csv("final_availability_after_fill.csv", index=False)

final_avail.head()


,station,file,pm25_pct,windspd_pct,winddir_pct
0,station_113_shadipur_delhi_cpcb_1hr,2021.csv,99.71,99.82,99.82
1,station_113_shadipur_delhi_cpcb_1hr,2022.csv,99.12,99.59,99.59
2,station_113_shadipur_delhi_cpcb_1hr,2023.csv,98.70,99.69,99.43
3,station_113_shadipur_delhi_cpcb_1hr,2024.csv,99.21,99.78,99.78
4,station_114_ihbas_dilshad_garden_delhi_cpcb_1hr,2021.csv,97.89,97.68,97.68
